# 03 · Calibration — refining setup parameters

Once you have a good sample/orientation match for one (or several) voxels,
you can use those to *refine the experimental geometry* (`Lsd`, `BC`, tilts,
wedge, etc.) so the rest of the dataset fits better.

`midas-nf-fitorientation` exposes two refinement modes:

| Mode | C analogue | Use case |
| --- | --- | --- |
| `fit_parameters_run(rowNr=…)` | `FitOrientationParameters` | refine on a single voxel |
| `fit_multipoint_run()`      | `FitOrientationParametersMultiPoint` | refine on a cluster of voxels (more stable) |

Both produce updated `Lsd`, `BC`, `tx/ty/tz`, `Wedge` values you paste back
into the param file before re-running the pipeline.

In [ ]:
import os, shutil, tempfile
from pathlib import Path

from midas_nf_fitorientation import fit_parameters_run, fit_multipoint_run

## Single-point refinement

Pass `row_nr=N` (0-based row in `grid.txt`) to refine using just that voxel.
Pick a voxel with high `Confidence` from a previous run.

In [ ]:
MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or '.')
AU_DIR = MIDAS_HOME / 'NF_HEDM/Example/sim'
ws = Path(tempfile.mkdtemp(prefix='nf_refine_'))
param = ws / 'test_ps_au.txt'
shutil.copy2(AU_DIR / 'test_ps_au.txt', param)
with open(param, 'a') as f:
    f.write(f'OutputDirectory {ws}\n')
print('workspace:', ws)

In [ ]:
# (Run the regular pipeline once first to produce SpotsInfo.bin and grid.txt;
# then come back here.) Refine on voxel #0:
# fit_parameters_run(str(param), row_nr=0, n_cpus=4, device='auto')

## Multi-point refinement

Recommended over single-point — more stable, less likely to pick a bad voxel.
The voxel set is configured via `GridPoints` in the param file (12 floats:
min/max for each of x, y, z, plus weighting parameters).

In [ ]:
# fit_multipoint_run(str(param), n_cpus=4, device='auto')

## CLI version

```bash
midas-nf-pipeline refine-params <param.txt> --row-nr 0
midas-nf-pipeline refine-params <param.txt> --multi-point
```